# Amparo -- M3 · S10: RAG de una pasada, evaluado por dentro

**Las rutas agenticas se retiraron el 2026-10-08 (C11).** Este notebook montaba
tres formas de responder -- una pasada, tool use (Lab A) y ReAct (Lab B) -- para
decidir con datos si un agente se justifica. Dos corridas independientes
(2026-09-27 y 2026-10-07, poblaciones distintas) dijeron lo mismo: el modelo
**nunca** decidio buscar por si mismo (en las 75 consultas la busqueda la forzo
el codigo) y las rutas agenticas costaban 2.2 veces la latencia sin ganar en
ninguna metrica. Ver `results/m3_s10_2026-10-07/scorecard.md` y
`docs/m3_decisiones_rag.md`, seccion 28. El codigo de esas rutas
(`tools/rag/tools.py`, `tools/rag/agentico.py`) y las salidas de la corrida del
07-10 quedan en la historia de git (commit `071c878`).

Lo que queda es el sistema que se entrega y su evaluacion:

| Fase | Que hace |
|---|---|
| 4 | Corre el RAG de una pasada (`pipeline.answer_query`) sobre el eval set, con latencia |
| 5 | **RAGAS** (Lab C, juez Groq): faithfulness, context precision/recall, answer relevancy |
| 5b | RAGAS sobre las busquedas A/B/C de `m3_s08_rag_avanzado.ipynb`: ¿que busqueda es mejor? |
| 5c | Guardia de rutas: ¿el RAG manda a la ventanilla correcta? |
| 6 | Registro en **Weights & Biases** |

La busqueda es la de `config.py` (`USE_HYBRID = USE_RERANK = False`, denso puro):
en la fase 5b del 07-10 el denso gano en context recall y faithfulness, y hybrid
y rerank no pagaron su costo.

**Requiere GPU** (T4 o superior) y el indice FAISS construido por
`m3_s07_rag_ingenuo.ipynb` en Drive.

**Secretos de Colab** (icono de llave, a la izquierda): `GROQ_API_KEY` para el
juez y `WANDB_API_KEY` para W&B. Nunca se escriben en el notebook.

In [18]:
# Rama del repo a clonar. Cambiala si el codigo del RAG avanzado todavia no esta
# en main (por ejemplo, mientras vive en una rama de PR).
RAMA = "m3.5"

import os

if not os.path.isdir("Amparo"):
    !git clone -b {RAMA} https://github.com/TomasPosada0626/Amparo.git
else:
    !cd Amparo && git fetch origin && git checkout {RAMA} && git pull

%cd Amparo

Cloning into 'Amparo'...
remote: Enumerating objects: 1136, done.
remote: Counting objects: 100% (782/782), done.
remote: Compressing objects: 100% (424/424), done.
remote: Total 1136 (delta 511), reused 578 (delta 342), pack-reused 354 (from 1)
Receiving objects: 100% (1136/1136), 4.62 MiB | 4.98 MiB/s, done.
Resolving deltas: 100% (700/700), done.
/content/Amparo/Amparo/Amparo


In [ ]:
import os, sys
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

REQUERIDOS = [
    "tools/rag/hybrid.py",
    "tools/rag/rerank.py",
    "tools/rag/verificacion.py",
    "tools/rag/pipeline.py",
    "data/eval_set.json",
]
faltan = [r for r in REQUERIDOS if not os.path.exists(r)]
if faltan:
    raise SystemExit(
        f"Falta en la rama '{RAMA}': {faltan}\n\n"
        "Este notebook corre contra el repo REMOTO. Si el codigo del RAG "
        "todavia esta solo en tu maquina, commitealo y pusheralo, o cambia RAMA."
    )
print("Repo OK.")

In [20]:
# Dependencias livianas del repo (incluye faiss-cpu y rank_bm25, el BM25 de la
# hybrid search).
!pip install -q -r requirements.txt

In [21]:
# Stack pesado: se instala aqui y no en requirements.txt, para no reemplazar el
# build de PyTorch con CUDA que Colab ya trae. sentence-transformers trae el
# cross-encoder del reranking; transformers alcanza para e5 y la generacion;
# peft/bitsandbytes solo si se genera con el adaptador LoRA.
!pip install -q -U transformers sentence-transformers peft bitsandbytes accelerate

## Fase 1 -- Cargar el indice, el BM25 y el generador

Mismo punto de partida que `m3_s08_rag_avanzado.ipynb`: el indice denso se copia
de Drive, el BM25 se construye una vez en memoria y Qwen2.5-7B se carga una sola
vez y se reusa en todas las rutas.

In [22]:
from google.colab import drive
import pathlib, shutil
from tools.rag import config

drive.mount("/content/drive")
origen = pathlib.Path(config.DRIVE_ROOT) / "rag"
config.ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
for nombre in ("rag_index.faiss", "rag_index_metadata.jsonl"):
    shutil.copy(origen / nombre, config.ARTIFACTS_DIR / nombre)
print("Indice copiado de Drive a artifacts/.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Indice copiado de Drive a artifacts/.


In [23]:
from tools.rag import pipeline
from tools.rag.hybrid import BM25Index

store = pipeline.load_index()
bm25 = BM25Index(store.metadata)   # una sola vez; se reusa en cada consulta
print(f"Indice denso: {len(store)} chunks | BM25: {len(bm25)} chunks")

Indice denso: 3420 chunks | BM25: 3420 chunks


In [24]:
# Amparo = modelo fine-tuneado de M1 (LoRA) + RAG: es el sistema que se entrega,
# asi que la corrida principal usa el adaptador. False sirve como ablacion
# ("¿el fine-tuning sigue aportando con RAG?"); los archivos de S10 quedan con
# sufijo _lora / _base, asi que las dos corridas no se pisan.
USE_LORA = True
model_bundle = pipeline.load_model(use_lora=USE_LORA)
print("Modelo cargado.")

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Modelo cargado.


## Fase 4 -- Corrida del RAG de una pasada sobre el eval set (con latencia)

Mismo eval set (`data/eval_set.json`: gold + adversariales) y la busqueda de
`config.py`. Se mide la latencia por consulta para reportarla junto a la calidad.

In [ ]:
import json, time
from tools.rag import config
from tools.evaluation import eval_set as eval_set_mod

registros = eval_set_mod.load_eval_set()
print(f"{len(registros)} registros ({len(eval_set_mod.gold_examples(registros))} gold, "
      f"{len(eval_set_mod.adversarial_examples(registros))} adversariales)")

# Una sola ruta (C11). La busqueda sale de config.py: si algun dia se cambia la
# configuracion, se cambia ahi y queda en el manifiesto, no escondida aca.
RUTAS = {
    "una_pasada": lambda q: pipeline.answer_query(
        q, store, use_lora=USE_LORA, bm25=bm25, model_bundle=model_bundle,
        use_hybrid=config.USE_HYBRID, use_rerank=config.USE_RERANK),
}

corridas, latencias = {}, {}
for ruta, responder in RUTAS.items():
    print(f"\nCorriendo ruta {ruta} ...")
    records, t0 = [], time.perf_counter()
    for i, reg in enumerate(registros, start=1):
        resultado = responder(reg["messages"][1]["content"])
        records.append(pipeline.to_eval_record(resultado, reg))
        if i % 10 == 0 or i == len(registros):
            print(f"  {i}/{len(registros)} -- {(time.perf_counter() - t0) / i:.1f} s/consulta")
    latencias[ruta] = (time.perf_counter() - t0) / len(registros)
    corridas[ruta] = records

In [ ]:
# Persistir en Drive para evaluar (RAGAS + harness) sin re-generar.
from collections import Counter
destino = pathlib.Path(config.DRIVE_ROOT) / "rag" / "corridas_s10"
destino.mkdir(parents=True, exist_ok=True)
etiqueta = "lora" if USE_LORA else "base"

for ruta, records in corridas.items():
    ruta_archivo = destino / f"eval_records_{ruta}_{etiqueta}.json"
    with open(ruta_archivo, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    print(f"  {ruta} -> {ruta_archivo}")

with open(destino / f"latencias_{etiqueta}.json", "w", encoding="utf-8") as f:
    json.dump(latencias, f, indent=2)

print("\nResumen:")
for ruta, records in corridas.items():
    sin_ctx = sum(1 for r in records if r["n_retrieved"] == 0)
    escapes = Counter((r.get("verificacion") or {}).get("escape_por_codigo") for r in records)
    corregidas = sum(1 for r in records if (r.get("verificacion") or {}).get("citas_rechazadas"))
    print(f"  {ruta:<11} {latencias[ruta]:6.1f} s/consulta | sin contexto: {sin_ctx:>2} | "
          f"escape por codigo: {sum(n for k, n in escapes.items() if k):>2} | "
          f"corregidas por la verificacion de citas: {corregidas:>2}")

## Fase 5 -- Lab C · RAGAS: evaluar el RAG por dentro

Cuatro metricas de 0 a 1 por caso, cada una comparando dos piezas:

| Metrica | Compara | Si sale baja... |
|---|---|---|
| faithfulness | respuesta vs contextos | el generador inventa o completa de memoria |
| context_precision | contextos vs pregunta | el retrieval trae ruido (rerank) |
| context_recall | referencia vs contextos | el retrieval no encuentra (hybrid, corpus) |
| answer_relevancy | respuesta vs pregunta | divaga o se niega a responder |

**Juez: Groq** (`openai/gpt-oss-120b`, el mismo de M2), de otra familia que Qwen,
para no repetir el sesgo de auto-preferencia que midio el M2. Las tres metricas de
contexto salen de una sola llamada por caso para cuidar el cupo diario de Groq; si
el cupo se acaba, la corrida queda guardada en un checkpoint en Drive y **se retoma
volviendo a correr esta celda** (mañana, o con la key de otro miembro del equipo).
Si Groq pide esperar (limite por minuto), el codigo espera y reintenta solo; si el
cupo diario se agoto, se detiene tras 3 fallos seguidos y avisa cuantos casos
faltan. Los casos que fallaron **no** se marcan como evaluados: se reintentan al
retomar. Mientras falten casos, el resumen dice que es PARCIAL.

Las metricas RAGAS se calculan sobre los casos **gold**. Los **adversariales** se
miden con la **prudencia**: escapa, o no cita articulos que no vio, no inventa
sentencias y no promete resultados (no todos esperan la frase de escape: ante una
amenaza, lo correcto es priorizar la seguridad). En gold tambien se cuenta cuantas
respuestas citan un articulo que el sistema no recupero (deberia ser 0).

Esta fase se puede correr en una sesion nueva: si `corridas` no esta en memoria,
se cargan de Drive (las guardo la fase 4). Necesita la Fase 1 (repo e indice).

In [ ]:
import json, os, pathlib
from google.colab import userdata
from tools.rag import config
from tools.evaluation import external_judge, ragas_metrics as rm

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
# El cliente de Groq se crea una vez con la primera key. Si se cambia el secreto
# (cupo agotado), volver a correr esta celda lo recrea con la key nueva.
external_judge._client = None

USE_LORA = globals().get("USE_LORA", True)
etiqueta = "lora" if USE_LORA else "base"
destino = pathlib.Path(config.DRIVE_ROOT) / "rag" / "corridas_s10"

if "corridas" not in globals():
    corridas = {}
    for ruta in ("una_pasada",):
        archivo = destino / f"eval_records_{ruta}_{etiqueta}.json"
        if archivo.exists():
            corridas[ruta] = json.loads(archivo.read_text(encoding="utf-8"))
    latencias = json.loads((destino / f"latencias_{etiqueta}.json").read_text(encoding="utf-8"))
    print("Corridas cargadas de Drive:", {r: len(v) for r, v in corridas.items()})

juez = rm.juez_groq()
embed = rm.embedder_e5()   # e5, el mismo embedding del RAG, para answer_relevancy

In [30]:
filas_ragas = {}
for ruta, records in corridas.items():
    print(f"\nRAGAS sobre la ruta {ruta} ...")
    checkpoint = destino / f"ragas_checkpoint_{ruta}_{etiqueta}.jsonl"
    filas_ragas[ruta] = rm.evaluar_corrida(records, juez=juez, embed=embed, checkpoint_path=checkpoint)

resumenes = rm.resumen([f for fs in filas_ragas.values() for f in fs])
escapes = rm.tasas_de_escape([r for rs in corridas.values() for r in rs])

with open(destino / f"ragas_resumen_{etiqueta}.json", "w", encoding="utf-8") as f:
    json.dump({"ragas": resumenes, "escape": escapes, "latencias": latencias}, f, ensure_ascii=False, indent=2)

print(f"\n{'ruta':<11}{'faith':>7}{'c.prec':>8}{'c.rec':>7}{'a.rel':>7}{'prud.adv':>9}{'esc.gold':>9}{'cita!gold':>10}{'s/cons':>8}{'parse!':>8}")
print("-" * 84)
fmt = lambda v: f"{v:.2f}" if isinstance(v, (int, float)) else "  - "
for ruta, r in resumenes.items():
    e = escapes.get(ruta, {})
    print(f"{ruta:<11}{fmt(r['faithfulness']):>7}{fmt(r['context_precision']):>8}{fmt(r['context_recall']):>7}"
          f"{fmt(r['answer_relevancy']):>7}{fmt(e.get('prudencia_en_adversariales')):>9}"
          f"{fmt(e.get('escape_en_gold')):>9}{fmt(e.get('citas_no_respaldadas_en_gold')):>10}"
          f"{fmt(latencias.get(ruta)):>8}{r['fallos_parseo']:>8}")
print("\nTokens del juez por ruta:", {r: v["tokens_juez"] for r, v in resumenes.items()})


RAGAS sobre la ruta una_pasada ...
[ragas] 5/45 -- tokens del juez en esta sesion: 13971
[ragas] 10/45 -- tokens del juez en esta sesion: 27513
[ragas] 15/45 -- tokens del juez en esta sesion: 39818
[ragas] 20/45 -- tokens del juez en esta sesion: 53401
[ragas] 25/45 -- tokens del juez en esta sesion: 66327
[ragas] 30/45 -- tokens del juez en esta sesion: 77658
[ragas] 35/45 -- tokens del juez en esta sesion: 89315
[ragas] 40/45 -- tokens del juez en esta sesion: 98952
[ragas] 45/45 -- tokens del juez en esta sesion: 107329

RAGAS sobre la ruta tool_use ...
[ragas] 5/45 -- tokens del juez en esta sesion: 13391
[ragas] 10/45 -- tokens del juez en esta sesion: 26969
[ragas] 15/45 -- tokens del juez en esta sesion: 39275
[ragas] 20/45 -- tokens del juez en esta sesion: 52527
[ragas] 25/45 -- tokens del juez en esta sesion: 64987
[ragas] 30/45 -- tokens del juez en esta sesion: 76655
[ragas] 35/45 -- tokens del juez en esta sesion: 88801
[ragas] 40/45 -- tokens del juez en esta sesion: 98

> **Como leer la tabla.** Las metricas son un diagnostico, no una nota. Context
> precision/recall bajas: el problema es la recuperacion. Faithfulness baja: el
> modelo no se apoya en el contexto. Answer relevancy baja: divaga o escapa de mas
> (mirar `esc.gold`). `cita!gold` > 0: alguna respuesta cita un articulo que no
> recupero, justo lo que Amparo no debe hacer. `prud.adv`: prudencia en los
> adversariales. `s/cons`: latencia por consulta. `parse!` cuenta los casos en que el
> juez no devolvio un JSON valido: no entran al promedio.

### Fase 5b -- RAGAS sobre las busquedas de S08: ¿que busqueda es mejor?

La S08 comparo tres formas de buscar (A denso, B + hybrid, C + rerank) pero no
tiene juez. Aqui, con el mismo juez Groq, se evaluan sus corridas guardadas en
Drive (`rag/corridas_abc/`). Las metricas que responden la pregunta de la S08 son
las de **contexto**:

- **context recall**: ¿la busqueda trae lo necesario para responder?
- **context precision**: ¿lo util queda arriba, sin ruido?

Faithfulness y answer relevancy completan el cuadro (que tan bien se uso lo que se
encontro).

**Cupo de Groq:** son otros ~50 casos gold por configuracion. Esta celda va
despues de la fase 5 para que, si el cupo se acaba, lo de S10 ya este evaluado; se retoma igual, volviendo a correrla. Si no quieren gastar cupo en esto,
`EVALUAR_ABC = False`. El checkpoint lleva una huella del archivo de S08: si S08 se
vuelve a correr, se evalua la corrida nueva y no se reusan resultados viejos.

**Ahorro:** la ruta "una pasada" de S10 es la misma configuracion que una de
las de S08 (con `config.py` por defecto, la A: LoRA, denso puro, mismo prompt).
Los casos con respuesta y
contextos identicos toman las notas ya calculadas en la fase 5, sin llamar al
juez; la celda imprime cuantos reuso.

In [ ]:
import hashlib

EVALUAR_ABC = True
abc_dir = pathlib.Path(config.DRIVE_ROOT) / "rag" / "corridas_abc"
corridas_abc, filas_abc, resumen_abc = {}, {}, {}

if EVALUAR_ABC:
    for letra, nombre in (("a", "A_denso"), ("b", "B_hybrid"), ("c", "C_rerank")):
        archivo = abc_dir / f"eval_records_config_{letra}.json"
        if not archivo.exists():
            continue
        recs = json.loads(archivo.read_text(encoding="utf-8"))
        for r in recs:
            r["sistema"] = nombre          # separa las configuraciones en el resumen
        corridas_abc[nombre] = recs
        huella = hashlib.md5(archivo.read_bytes()).hexdigest()[:8]
        print(f"\nRAGAS sobre {nombre} ({len(recs)} registros, huella {huella}) ...")
        filas_abc[nombre] = rm.evaluar_corrida(
            recs, juez=juez, embed=embed,
            checkpoint_path=abc_dir / f"ragas_checkpoint_{nombre}_{huella}.jsonl",
            # "una_pasada" de S10 es una de las configuraciones de S08: los casos
            # identicos (misma respuesta y contextos) toman las notas ya
            # calculadas, sin gastar cupo de Groq.
            reusar=[f for fs in filas_ragas.values() for f in fs])

    if not corridas_abc:
        print("No hay corridas de S08 en Drive: correr m3_s08_rag_avanzado.ipynb (fase 4).")
    else:
        resumen_abc = rm.resumen([f for fs in filas_abc.values() for f in fs])
        with open(abc_dir / "ragas_resumen_abc.json", "w", encoding="utf-8") as f:
            json.dump(resumen_abc, f, ensure_ascii=False, indent=2)

        print(f"\n¿Que busqueda es mejor? (RAGAS, juez {external_judge.GROQ_JUDGE_MODEL if 'external_judge' in globals() else 'Groq'})")
        print(f"{'config':<10}{'c.recall':>9}{'c.prec':>8}{'faith':>7}{'a.rel':>7}{'parse!':>8}")
        print("-" * 49)
        for nombre, r in resumen_abc.items():
            print(f"{nombre:<10}{fmt(r['context_recall']):>9}{fmt(r['context_precision']):>8}"
                  f"{fmt(r['faithfulness']):>7}{fmt(r['answer_relevancy']):>7}{r['fallos_parseo']:>8}")

### Fase 5c -- Guardia de rutas: ¿el RAG manda a la ventanilla correcta?

Es la pregunta concreta que M3 responde antes de decidir si hay que reentrenar
M1: el fine-tuning **empeoro** las rutas y la duda es si recuperar lo corrige.

La comparacion se hace contra las salidas que M2 guardo sobre **estas mismas 75
preguntas** (`results/m2_2026-10-06/eval_set_*.jsonl`), no contra el 7.4 % de la
validacion: son poblaciones distintas y mezclarlas no dice nada.

Tres cuidados que cambian el numero:

- **Gold y adversariales por separado.** Los 30 adversariales piden abstenerse,
  asi que mezclarlos con los gold diluye la medicion.
- **Los escapes no cuentan.** Una respuesta que no orienta a ningun lado no
  puede equivocarse de ventanilla; contarla como acierto infla la cifra, y el
  RAG escapa mas que el modelo solo.
- **Con intervalo.** Con n=45 y n=30, una diferencia de un caso no es un
  resultado.

Se usa la misma guardia que M2 (`tools/evaluation/rutas.py`), asi que las
cifras son comparables. No gasta cupo de Groq.


In [33]:
import json
from pathlib import Path

from tools.evaluation import dataset as m1_dataset, rutas
from tools.evaluation.estadistica import proporcion
from tools.evaluation.ragas_metrics import es_valvula_de_escape

# El mapa de contexto se aprende del TRAIN de M1: en que temas aparece cada
# concepto. Una respuesta que usa un concepto donde el train nunca lo usa se
# marca "fuera de contexto".
_train, _val = m1_dataset.stratified_split(m1_dataset.load_records())
mapa_rutas = rutas.construir_mapa(_train)

# Salidas de M2 sobre LAS MISMAS 75 preguntas: es la unica comparacion valida.
# Si no estan, la celda lo dice en vez de comparar contra validacion.
M2_DIR = Path("results/m2_2026-10-06")
sin_rag = {}
for etiqueta, archivo in (("base sin RAG", "eval_set_baseline_results.jsonl"),
                          ("fine-tuned sin RAG", "eval_set_finetuned_results.jsonl")):
    ruta = M2_DIR / archivo
    if ruta.exists():
        sin_rag[etiqueta] = [json.loads(l) for l in ruta.read_text(encoding="utf-8").splitlines() if l.strip()]
    else:
        print(f"FALTA {ruta}: sin eso no hay comparacion valida contra el modelo sin RAG.")

_tipo_de = {r["id"]: r["tipo"] for r in registros}


def _filas(items, *, campo_respuesta, campo_pregunta, campo_categoria):
    return [{"id": r["id"], "query": r[campo_pregunta], "category": r[campo_categoria],
             "generated": r[campo_respuesta]} for r in items]


def _medir(etiqueta, filas):
    """Ruta incorrecta por tipo, excluyendo escapes, con IC de Wilson."""
    salida = {}
    for tipo in ("gold", "adversarial"):
        del_tipo = [f for f in filas if _tipo_de.get(f["id"]) == tipo]
        # Un escape no orienta a ninguna ventanilla: no puede equivocarse.
        orientan = [f for f in del_tipo if not es_valvula_de_escape(f["generated"])]
        malas = sum(1 for f in orientan if rutas.rutas_incorrectas(f["generated"], f["query"]))
        ic = proporcion(malas, len(orientan)) if orientan else None
        salida[tipo] = {
            "n_total": len(del_tipo), "n_orientan": len(orientan), "malas": malas,
            "pct": round(100 * malas / len(orientan), 1) if orientan else None,
            "ic95": [round(100 * ic.bajo, 1), round(100 * ic.alto, 1)] if ic else None,
        }
    return etiqueta, salida


mediciones = []
# Calibracion: las referencias del propio eval set, correctas por construccion.
mediciones.append(_medir("referencias (calibracion)", _filas(
    [{"id": r["id"], "q": r["messages"][1]["content"], "c": r["category"],
      "a": r["messages"][2]["content"]} for r in registros],
    campo_respuesta="a", campo_pregunta="q", campo_categoria="c")))
for etiqueta, filas_sin_rag in sin_rag.items():
    mediciones.append(_medir(etiqueta, _filas(
        filas_sin_rag, campo_respuesta="generated", campo_pregunta="query", campo_categoria="category")))
for ruta_nombre, records in corridas.items():
    mediciones.append(_medir(f"{ruta_nombre} (con RAG)", _filas(
        records, campo_respuesta="answer", campo_pregunta="question", campo_categoria="category")))

print(f"{'sistema':30s}{'gold: ruta incorrecta':>30}{'adversarial':>28}")
print("-" * 88)
for etiqueta, d in mediciones:
    def _celda(t):
        x = d[t]
        if x["pct"] is None:
            return "sin casos"
        return f"{x['malas']}/{x['n_orientan']} = {x['pct']}% [{x['ic95'][0]}-{x['ic95'][1]}]"
    print(f"{etiqueta:30s}{_celda('gold'):>30}{_celda('adversarial'):>28}")
print("-" * 88)
print("Excluidos los escapes: una respuesta que no orienta no puede equivocarse de ventanilla.")
print("n_orientan por sistema:", {e: {t: d[t]["n_orientan"] for t in d} for e, d in mediciones})

# Es el resultado que responde la pregunta de M3 (¿el RAG corrige las rutas?) y
# hasta ahora era el unico que no quedaba en disco: la celda solo imprimia.
# Se guarda con los n y los intervalos, no solo los porcentajes, porque con 0-2
# errores sobre 36-45 casos el intervalo es la mitad del resultado.
_destino_5c = Path(config.DRIVE_ROOT) / "rag" / "corridas_s10"
_destino_5c.mkdir(parents=True, exist_ok=True)
with open(_destino_5c / "rutas_5c.json", "w", encoding="utf-8") as f:
    json.dump(dict(mediciones), f, ensure_ascii=False, indent=2)
print("\nTabla 5c ->", _destino_5c / "rutas_5c.json")

sistema                                gold: ruta incorrecta                 adversarial
----------------------------------------------------------------------------------------
referencias (calibracion)              0/45 = 0.0% [0.0-7.9]      0/30 = 0.0% [0.0-11.3]
base sin RAG                          1/45 = 2.2% [0.4-11.6]      0/30 = 0.0% [0.0-11.3]
fine-tuned sin RAG                    2/45 = 4.4% [1.2-14.8]      1/30 = 3.3% [0.6-16.7]
una_pasada (con RAG)                   0/40 = 0.0% [0.0-8.8]      0/26 = 0.0% [0.0-12.9]
tool_use (con RAG)                    1/40 = 2.5% [0.4-12.9]      0/27 = 0.0% [0.0-12.5]
react (con RAG)                       1/36 = 2.8% [0.5-14.2]      0/25 = 0.0% [0.0-13.3]
----------------------------------------------------------------------------------------
Excluidos los escapes: una respuesta que no orienta no puede equivocarse de ventanilla.
n_orientan por sistema: {'referencias (calibracion)': {'gold': 45, 'adversarial': 30}, 'base sin RAG': {'gold':

## Fase 6 -- Registrar en Weights & Biases

La corrida queda en el proyecto `amparo-rag` (en la cuenta del equipo), con sus
promedios y una tabla por caso. Todas las corridas de esta sesion comparten un
`group`, para compararlas con futuras versiones del sistema.

Si no hay `WANDB_API_KEY` en los secretos, corre offline y guarda en `wandb/`
(se sube despues con `wandb sync`).

**Si la fase 5b se corre otro dia** (el cupo de Groq no alcanzo): correr la fase 6
ahora solo registra S10. Despues, en la sesion en que se complete la 5b, correr de
nuevo las celdas de la fase 5 (leen el checkpoint, no gastan cupo), la 5b y esta
celda con `REGISTRAR_S10 = False`: asi solo se suben las busquedas A/B/C.

In [ ]:
%pip install -q wandb
import datetime, wandb
from tools.evaluation import tracking, external_judge

WANDB_ENTITY = "tomasposada67-universidad-eafit"   # cuenta del equipo (la de M1)
WANDB_PROJECT = "amparo-rag"

try:
    wandb_key = userdata.get("WANDB_API_KEY")
except Exception:
    wandb_key = None
print("W&B en modo:", tracking.modo_de_ejecucion(wandb_key))

grupo = f"s10-{etiqueta}-{datetime.date.today().isoformat()}"
config_run = {
    "generador": "Qwen/Qwen2.5-7B-Instruct",
    "use_lora": USE_LORA,
    "retrieval": {(False, False): "denso (config A)", (True, False): "hybrid (config B)",
                  (True, True): "hybrid + rerank (config C)"}.get((config.USE_HYBRID, config.USE_RERANK),
                                                                   f"hybrid={config.USE_HYBRID} rerank={config.USE_RERANK}"),
    "top_k": config.TOP_K,
    "min_score": config.RETRIEVAL_MIN_SCORE,
    "juez_ragas": external_judge.GROQ_JUDGE_MODEL,
    "n_registros": len(next(iter(corridas.values()))),
}

# Si ya se registraron las rutas de S10 en una sesion anterior y ahora solo se
# corrio la fase 5b (cupo de Groq), poner False para no duplicarlas en W&B.
REGISTRAR_S10 = True

for ruta, records in (corridas.items() if REGISTRAR_S10 else []):
    url = tracking.registrar_ruta(
        wandb, sistema=ruta, resumen_ragas=resumenes.get(ruta, {}), escape=escapes.get(ruta, {}),
        latencia_s=latencias.get(ruta), filas_ragas=filas_ragas.get(ruta, []), records=records,
        config=config_run, entity=WANDB_ENTITY, project=WANDB_PROJECT, grupo=grupo,
    )
    print(f"  {ruta}: {url or '(offline)'}")

if REGISTRAR_S10:
    url = tracking.registrar_comparativa(
        wandb, resumenes=resumenes, escapes=escapes, latencias=latencias,
        config=config_run, entity=WANDB_ENTITY, project=WANDB_PROJECT, grupo=grupo,
    )
    print("Comparativa:", url or "(offline)")

# Busquedas de S08 (fase 5b), en su propio grupo
if globals().get("corridas_abc"):
    escapes_abc = rm.tasas_de_escape([r for rs in corridas_abc.values() for r in rs])
    grupo_abc = f"s08-abc-{datetime.date.today().isoformat()}"
    for nombre, recs in corridas_abc.items():
        url = tracking.registrar_ruta(
            wandb, sistema=nombre, resumen_ragas=resumen_abc.get(nombre, {}),
            escape=escapes_abc.get(nombre, {}), latencia_s=None, filas_ragas=filas_abc.get(nombre, []),
            records=recs, config={**config_run, "experimento": "S08 A/B/C (busqueda)"},
            entity=WANDB_ENTITY, project=WANDB_PROJECT, grupo=grupo_abc,
        )
        print(f"  {nombre}: {url or '(offline)'}")
    tracking.registrar_comparativa(
        wandb, resumenes=resumen_abc, escapes=escapes_abc, latencias={},
        config={**config_run, "experimento": "S08 A/B/C (busqueda)"},
        entity=WANDB_ENTITY, project=WANDB_PROJECT, grupo=grupo_abc,
    )

---
### Contenido

1. La corrida del RAG de una pasada en formato RAGAS, con su latencia (Fase 4),
   en `corridas_s10/` de Drive.
2. **RAGAS** (`tools/evaluation/ragas_metrics.py`, juez Groq), mas la tasa de
   escape en adversariales y gold (Fase 5), en `corridas_s10/ragas_resumen_*.json`.
3. RAGAS sobre las busquedas A/B/C de S08 (Fase 5b) y la guardia de rutas (Fase 5c).
4. Las corridas registradas en **W&B** (`tools/evaluation/tracking.py`, Fase 6).

*Amparo · M3 · S10.*

## Manifiesto de la corrida

Guarda junto a los resultados **con que** se produjeron: commit, versiones de
las librerias y huellas del indice, el corpus, el dataset, el eval set y el
adaptador.

Huellas y no fechas: dos corridas pueden decir las dos "2026-10-07" y haber
medido sobre indices distintos. Si declaran el mismo hash, midieron sobre lo
mismo.


In [35]:
# Manifiesto de la corrida: commit, versiones y huellas del indice, el corpus,
# el dataset, el eval set y el adaptador. Sin esto no se sabe con que salio cada
# cifra -- el adaptador de Drive ya se sobrescribio una vez (6 de octubre) y las
# corridas anteriores a esa fecha quedaron sin forma de reproducirse.
#
# Lee las variables del notebook con globals().get(): un manifiesto describe la
# corrida, no puede tumbarla. Si falta alguna, queda en None y se anota.
import pathlib
import subprocess

from tools.rag import manifiesto

_hardware = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
).stdout.strip()

_store = globals().get("store")
_registros = globals().get("registros")

_manifiesto = manifiesto.construir(
    "s10",
    use_lora=bool(globals().get("USE_LORA", False)),
    n_chunks=len(_store.metadata) if _store is not None else None,
    n_eval_set=len(_registros) if _registros is not None else None,
    hardware=_hardware,
)
_destino = pathlib.Path(config.DRIVE_ROOT) / "rag" / "corridas_s10"
print("Manifiesto:", _manifiesto.guardar(_destino))
print(f"  commit {_manifiesto.git_commit[:8]} | indice {_manifiesto.hash_indice} "
      f"| adaptador {_manifiesto.hash_adaptador} | eval set {_manifiesto.hash_eval_set}")
print(f"  chunks {_manifiesto.n_chunks} | casos {_manifiesto.n_eval_set}")


Manifiesto: /content/drive/MyDrive/Colab Notebooks/Amparo/rag/corridas_s10/run_manifest.json
  commit 21cd93c5 | indice 4269a04083d935fd | adaptador 130d8f65b6e34746 | eval set a5151999c2095d00
  chunks 3420 | casos 75
